# Solution F2 · Rare classes: weights, undersampling, oversampling or just the threshold?

**Type**: Compare

## Objective

Compare five strategies for a rare event (about 3.5% positives): base model with threshold 0.5, `class_weight="balanced"`, random undersampling and oversampling (on the training set only) and base model with **threshold chosen
in cross-validation**. Measure average precision, AUC, F1 and Brier score on 20,000 new examples, averaged over 10 datasets.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(seed):
    X, y = make_classification(n_samples=23000, n_features=12, n_informative=5, n_redundant=0, weights=[0.97], class_sep=0.9, flip_y=0.01, random_state=seed)
    return X[:3000], y[:3000], X[3000:], y[3000:]


def model(**kw):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, **kw))

## Commented solution

In [ ]:
def best_threshold(y, p):
    ts = np.linspace(0.02, 0.9, 89)
    return ts[int(np.argmax([f1_score(y, p >= t) for t in ts]))]

In [ ]:
rows = []
for seed in range(10):
    X_tr, y_tr, X_te, y_te = generate(seed)
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y_tr == 1)[0], np.where(y_tr == 0)[0]

    def row(name, m, X_fit, y_fit, threshold=0.5):
        p = m.fit(X_fit, y_fit).predict_proba(X_te)[:, 1]
        rows.append({"seed": seed, "strategy": name, "ap": average_precision_score(y_te, p), "auc": roc_auc_score(y_te, p),
                     "f1": f1_score(y_te, p >= threshold), "recall": recall_score(y_te, p >= threshold),
                     "precision": precision_score(y_te, p >= threshold, zero_division=0), "brier": brier_score_loss(y_te, p), "mean_p": p.mean()})

    row("base", model(), X_tr, y_tr)
    row("balanced", model(class_weight="balanced"), X_tr, y_tr)
    u = np.r_[pos, rng.choice(neg, len(pos), replace=False)]
    row("undersampling", model(), X_tr[u], y_tr[u])
    o = np.r_[neg, rng.choice(pos, len(neg))]
    row("oversampling", model(), X_tr[o], y_tr[o])
    oof = cross_val_predict(model(), X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    row("cv threshold", model(), X_tr, y_tr, threshold=best_threshold(y_tr, oof))
table = pd.DataFrame(rows).groupby("strategy").mean().drop(columns="seed").loc[["base", "balanced", "undersampling", "oversampling", "cv threshold"]]
table.round(3)

In [ ]:
best_f1 = table["f1"].idxmax()
best_brier = table["brier"].idxmin()
balanced_mean_p = table.loc["balanced", "mean_p"]
print(f"best F1: {best_f1} | best Brier: {best_brier} | mean probability predicted by 'balanced': {balanced_mean_p:.3f} (real prevalence about 0.035)")

## Check

In [ ]:
with verify():
    check(set(table.index) == {"base", "balanced", "undersampling", "oversampling", "cv threshold"} and list(table.columns) == ["ap", "auc", "f1", "recall", "precision", "brier", "mean_p"], "five strategies and seven metrics", "columns: ap, auc, f1, recall, precision, brier, mean_p")
    check(best_f1 == "cv threshold" and table.loc["cv threshold", "f1"] > table.loc["base", "f1"] + 0.08, "the threshold chosen in cross-validation has the best F1 of all (and by over 0.08 compared to base)", "compare the f1 column")
    check(table.loc[["balanced", "undersampling", "oversampling"], "recall"].min() > 0.6 and table.loc[["balanced", "undersampling", "oversampling"], "precision"].max() < 0.2, "the three rebalancings have high recall (above 0.6) and low precision (below 0.2)", "they are shifts of the effective threshold")
    check(abs(table["auc"].max() - table["auc"].min()) < 0.03, "the AUC is practically the same (difference below 0.03)", "rebalancing does not change the ordering much")
    check(table.loc["base", "ap"] > table.loc[["balanced", "undersampling", "oversampling"], "ap"].max() + 0.05, "the average precision of the base model is better by 0.05 than any rebalancing", "AP: base model against rebalanced ones")
    check(best_brier in ("base", "cv threshold") and balanced_mean_p > 0.25 and table.loc["base", "mean_p"] < 0.05, "the best Brier is that of the base model; `balanced` predicts on average over 0.25 against a prevalence of 3.5%", "minimum brier and mean_p")
done()

## What you learned

- **No rebalancing improves the ordering:** the AUC is almost equal (0.81 against 0.82) and the **average precision gets worse** (0.39 against 0.25-0.29), because weights and samplings reduce the information
  (undersampling throws away 90% of the negatives) or duplicate it (oversampling).
- The three rebalancings are, in practice, **a threshold change in disguise**: recall 0.73, precision 0.10-0.11 and low F1 (0.17-0.20) at threshold 0.5. They move the working point along the curve without changing it, and in the wrong direction for the F1.
- **The probabilities get ruined**: the mean predicted by `balanced` is 0.33 against a prevalence of 3.5%, and the Brier is six times worse (0.153 against 0.026). If you need probabilities (costs, ranking, recalibration) it is damage.
- The best strategy for F1 is the simplest one: **base model + threshold chosen in cross-validation** (F1 0.43 against 0.29 for the base at threshold 0.5 and 0.17-0.20 for the rebalancings), without distorting the probabilities. Rebalancing
  can help with models that ignore the rare class (very deep trees, unregulated boosting) but it must be measured, not taken for granted.

## Common pitfall

Applying undersampling or oversampling "because the classes are imbalanced" and judging the result with accuracy or with the recall at threshold 0.5. Before rebalancing, try the threshold: if the base model ranks well (high AUC),
moving the threshold is almost always enough and costs nothing.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You have a dataset with 1% positives. Where do you start?</summary>

From the base model without rebalancing, evaluated with average precision and precision-recall curves (not accuracy), and from the threshold chosen on the costs or in cross-validation. Only if the results are insufficient do I try class weights
or sampling, always inside the validation (only on the training part of every fold) and comparing the probabilities and the calibration, not just the recall. With tree models I also consider the hyperparameters that control
depth and minimum leaves, which often matter more than rebalancing.

</details>